# 17｜自然語言處理_詞嵌入與注意力

由詞嵌入計算 scaled dot-product attention，直接讀權重矩陣。

從上到下執行程式格。第一格安裝缺少的套件，第二格準備本章所需設定。

In [ ]:
# 第 1 格：安裝本 Notebook 實際使用的套件。
import importlib.util, subprocess, sys
_needed = {'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.0,<3', 'matplotlib': 'matplotlib>=3.7,<4'}
_missing = [spec for module, spec in _needed.items() if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_missing])


In [ ]:
# 第 2 格：本 Notebook 的輸出位置與繪圖設定。
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
ROOT = Path.cwd()
OUT = ROOT / ("outputs" if (ROOT / "pyproject.toml").exists() else "mlcourse_outputs")
plt.rcParams.update({"figure.figsize": (8, 4.5), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.2, "font.size": 11})

def savefig(name):
    folder = OUT / "figures"
    folder.mkdir(parents=True, exist_ok=True)
    plt.gcf().savefig(folder / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.gcf().savefig(folder / f"{name}.svg", bbox_inches="tight")
    plt.show()

def report(name, values):
    folder = OUT / "reports"
    folder.mkdir(parents=True, exist_ok=True)
    (folder / f"{name}.json").write_text(
        json.dumps(values, ensure_ascii=False, indent=2, default=float) + "\n")


## 課堂小實驗

In [ ]:
tokens = ['我', '喜歡', '機器', '學習']
E = np.array([[1.,0.,0.], [.8,.2,0.], [0.,.8,.4], [0.,.6,.8]])
Q = K = V = E
scores = Q @ K.T / np.sqrt(E.shape[1])
weights = np.exp(scores - scores.max(axis=1, keepdims=True))
weights /= weights.sum(axis=1, keepdims=True)
context = weights @ V
pd.DataFrame(weights, index=tokens, columns=tokens).round(3)

## 執行結果圖

In [ ]:
plt.imshow(weights, cmap='Blues', vmin=0, vmax=weights.max())
plt.xticks(range(4), tokens); plt.yticks(range(4), tokens)
plt.xlabel('key token'); plt.ylabel('query token'); plt.title('Scaled dot-product attention weights')
for (i,j), value in np.ndenumerate(weights): plt.text(j, i, f'{value:.2f}', ha='center', va='center')
savefig('17_attention_demo')
report('nlp_attention', {'tokens': tokens, 'weights': weights.tolist(), 'context': context.tolist()})

## 練習：改動一個參數

先預測結果，再修改程式中的一個參數並重跑；比較圖表或數值，說明差異。